# Active Learning: Data-Efficient Machine Learning

**Active Learning** is a paradigm where the model intelligently selects which data points to label, dramatically reducing annotation costs.

## What We'll Learn

- The annotation bottleneck in machine learning
- How active learning reduces labeling costs by 10-100x
- Query strategies: uncertainty, committee, diversity
- Building complete active learning loops
- Deep active learning with neural networks
- Real-world applications and practical considerations

## Why It Matters

Labeling data is expensive:
- Medical imaging: $50-200 per image
- Autonomous driving: hours per edge case
- Domain-specific NLP: expert annotators required

Active learning makes the most of limited annotation budgets by asking: **"Which samples, if labeled, would improve the model the most?"**

## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances
from collections import defaultdict
import copy

from aiml_notebooks import (
    create_dataset,
    get_device,
    set_seed,
    MNIST_MEAN,
    MNIST_STD,
)

%load_ext autoreload
%autoreload 2

## Set Random Seed and Device

In [ ]:
set_seed(42)
device = get_device()
print(f"Using device: {device}")

## 1. The Annotation Bottleneck

In traditional supervised learning, we need **massive labeled datasets**:

- ImageNet: 1.2M labeled images
- Took thousands of person-hours
- Cost millions of dollars

But what if we could achieve similar performance with **10x or 100x fewer labels**?

That's the promise of active learning.

### The Key Insight

Not all data points are equally informative:

- **Easy examples**: Model already confident → labeling adds little value
- **Hard examples**: Model uncertain → labeling teaches new patterns
- **Redundant examples**: Similar to labeled data → no new information
- **Diverse examples**: Cover unexplored regions → maximize coverage

Active learning intelligently selects the **most informative** samples to label.

## 2. The Active Learning Loop

Active learning is an iterative process:

```
1. Start with small labeled set L and large unlabeled pool U
2. Train model on L
3. Query strategy: Select k most informative samples from U
4. Oracle (human) labels selected samples
5. Move selected samples from U to L
6. Repeat until budget exhausted or performance target reached
```

The **query strategy** is the key component that determines which samples to label.

### Load MNIST Dataset

We'll use MNIST to demonstrate active learning. We'll simulate the oracle by using the true labels.

In [ ]:
# Load full MNIST dataset
# For vision datasets, create_dataset returns (train_dataset, test_dataset) when no splits specified
train_dataset, test_dataset = create_dataset("mnist")

# We'll use train_dataset as our full pool (training + unlabeled)
full_dataset = train_dataset

print(f"Training pool size: {len(full_dataset)}")
print(f"Test set size: {len(test_dataset)}")

### Create Initial Splits

We'll start with:
- **Labeled set**: 100 samples (initial seed)
- **Unlabeled pool**: 9,900 samples
- **Validation set**: 10,000 samples (for unbiased evaluation)

This simulates a realistic scenario where we have very limited labeled data.

In [ ]:
# Create initial random split
n_initial = 100  # Start with only 100 labeled samples
n_pool = 9900    # Keep 9,900 unlabeled

# Random indices
indices = torch.randperm(len(full_dataset)).tolist()
labeled_indices = indices[:n_initial]
unlabeled_indices = indices[n_initial:n_initial + n_pool]

print(f"Initial labeled set: {len(labeled_indices)} samples")
print(f"Unlabeled pool: {len(unlabeled_indices)} samples")
print(f"Test set: {len(test_dataset)} samples")

### Define Simple CNN Classifier

We'll use a simple CNN for MNIST classification.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10, dropout=0.5):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, num_classes)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, return_features=False):
        # Conv layers
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        
        # Flatten
        features = x.view(x.size(0), -1)
        
        # FC layers
        x = F.relu(self.fc1(features))
        x = self.dropout(x)
        logits = self.fc2(x)
        
        if return_features:
            return logits, features
        return logits

model = SimpleCNN().to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters())}")

### Training and Evaluation Functions

Helper functions to train and evaluate the model.

In [ ]:
def train_model(model, train_indices, epochs=10, batch_size=32, lr=0.001):
    """Train model on labeled data."""
    train_subset = Subset(full_dataset, train_indices)
    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
    
    return total_loss / len(train_loader)

def evaluate_model(model, test_loader):
    """Evaluate model accuracy."""
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    return correct / total

test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
print("Training and evaluation functions ready")

## 3. Query Strategies

The heart of active learning is the **query strategy** - how to select which samples to label next.

We'll implement three main families:

1. **Uncertainty Sampling**: Select samples where model is most uncertain
2. **Query-by-Committee**: Select samples where ensemble of models disagree
3. **Diversity Sampling**: Select samples that cover the data space

Let's start with uncertainty sampling.

### 3.1 Uncertainty Sampling: Least Confidence

**Idea**: Select samples where the model has the **lowest confidence** in its prediction.

For a prediction $p = [p_1, p_2, ..., p_K]$, confidence is:

$$\text{confidence} = \max_k p_k$$

We select samples with **minimum confidence** (i.e., maximum uncertainty).

In [ ]:
def least_confidence_sampling(model, unlabeled_indices, n_samples=10, batch_size=128):
    """Select samples with lowest prediction confidence."""
    model.eval()
    
    # Create dataloader for unlabeled pool
    unlabeled_subset = Subset(full_dataset, unlabeled_indices)
    unlabeled_loader = DataLoader(unlabeled_subset, batch_size=batch_size, shuffle=False)
    
    # Compute confidence for each sample
    confidences = []
    with torch.no_grad():
        for images, _ in unlabeled_loader:
            images = images.to(device)
            outputs = model(images)
            probs = F.softmax(outputs, dim=1)
            # Confidence = max probability
            max_probs, _ = torch.max(probs, dim=1)
            confidences.extend(max_probs.cpu().numpy())
    
    confidences = np.array(confidences)
    
    # Select samples with lowest confidence
    uncertain_indices = np.argsort(confidences)[:n_samples]
    selected_indices = [unlabeled_indices[i] for i in uncertain_indices]
    
    return selected_indices, confidences[uncertain_indices]

print("Least confidence sampling ready")

### 3.2 Uncertainty Sampling: Margin Sampling

**Idea**: Select samples where the margin between the top 2 predictions is **smallest**.

Margin:

$$\text{margin} = p_{\text{1st}} - p_{\text{2nd}}$$

Small margin = model is confused between two classes.

In [ ]:
def margin_sampling(model, unlabeled_indices, n_samples=10, batch_size=128):
    """Select samples with smallest margin between top 2 predictions."""
    model.eval()
    
    unlabeled_subset = Subset(full_dataset, unlabeled_indices)
    unlabeled_loader = DataLoader(unlabeled_subset, batch_size=batch_size, shuffle=False)
    
    margins = []
    with torch.no_grad():
        for images, _ in unlabeled_loader:
            images = images.to(device)
            outputs = model(images)
            probs = F.softmax(outputs, dim=1)
            
            # Get top 2 probabilities
            top2_probs, _ = torch.topk(probs, 2, dim=1)
            # Margin = difference between 1st and 2nd
            batch_margins = top2_probs[:, 0] - top2_probs[:, 1]
            margins.extend(batch_margins.cpu().numpy())
    
    margins = np.array(margins)
    
    # Select samples with smallest margin
    uncertain_indices = np.argsort(margins)[:n_samples]
    selected_indices = [unlabeled_indices[i] for i in uncertain_indices]
    
    return selected_indices, margins[uncertain_indices]

print("Margin sampling ready")

### 3.3 Uncertainty Sampling: Entropy-based

**Idea**: Select samples with **highest prediction entropy**.

Entropy measures the "spread" of the probability distribution:

$$H(p) = -\sum_{k=1}^K p_k \log p_k$$

- High entropy = uncertain (uniform distribution)
- Low entropy = confident (peaked distribution)

In [ ]:
def entropy_sampling(model, unlabeled_indices, n_samples=10, batch_size=128):
    """Select samples with highest prediction entropy."""
    model.eval()
    
    unlabeled_subset = Subset(full_dataset, unlabeled_indices)
    unlabeled_loader = DataLoader(unlabeled_subset, batch_size=batch_size, shuffle=False)
    
    entropies = []
    with torch.no_grad():
        for images, _ in unlabeled_loader:
            images = images.to(device)
            outputs = model(images)
            probs = F.softmax(outputs, dim=1)
            
            # Compute entropy: -sum(p * log(p))
            log_probs = torch.log(probs + 1e-10)  # Add epsilon for numerical stability
            batch_entropies = -(probs * log_probs).sum(dim=1)
            entropies.extend(batch_entropies.cpu().numpy())
    
    entropies = np.array(entropies)
    
    # Select samples with highest entropy
    uncertain_indices = np.argsort(entropies)[::-1][:n_samples]  # Descending order
    selected_indices = [unlabeled_indices[i] for i in uncertain_indices]
    
    return selected_indices, entropies[uncertain_indices]

print("Entropy sampling ready")

### Compare Uncertainty Measures

Let's visualize what each uncertainty measure captures on a few examples.

In [ ]:
# Train initial model
model = SimpleCNN().to(device)
train_model(model, labeled_indices, epochs=5)

# Sample a few unlabeled examples
sample_indices = unlabeled_indices[:1000]

# Get predictions
model.eval()
with torch.no_grad():
    sample_subset = Subset(full_dataset, sample_indices)
    sample_loader = DataLoader(sample_subset, batch_size=128, shuffle=False)
    
    all_probs = []
    for images, _ in sample_loader:
        images = images.to(device)
        outputs = model(images)
        probs = F.softmax(outputs, dim=1)
        all_probs.append(probs.cpu())
    
    all_probs = torch.cat(all_probs, dim=0)

# Compute uncertainty measures
confidences = torch.max(all_probs, dim=1)[0].numpy()
top2 = torch.topk(all_probs, 2, dim=1)[0]
margins = (top2[:, 0] - top2[:, 1]).numpy()
entropies = -(all_probs * torch.log(all_probs + 1e-10)).sum(dim=1).numpy()

# Plot distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(confidences, bins=50, alpha=0.7, edgecolor='black')
axes[0].set_xlabel('Confidence (max probability)')
axes[0].set_ylabel('Count')
axes[0].set_title('Least Confidence')
axes[0].axvline(confidences.mean(), color='red', linestyle='--', label=f'Mean: {confidences.mean():.3f}')
axes[0].legend()

axes[1].hist(margins, bins=50, alpha=0.7, edgecolor='black')
axes[1].set_xlabel('Margin (top1 - top2)')
axes[1].set_ylabel('Count')
axes[1].set_title('Margin Sampling')
axes[1].axvline(margins.mean(), color='red', linestyle='--', label=f'Mean: {margins.mean():.3f}')
axes[1].legend()

axes[2].hist(entropies, bins=50, alpha=0.7, edgecolor='black')
axes[2].set_xlabel('Entropy')
axes[2].set_ylabel('Count')
axes[2].set_title('Entropy Sampling')
axes[2].axvline(entropies.mean(), color='red', linestyle='--', label=f'Mean: {entropies.mean():.3f}')
axes[2].legend()

plt.tight_layout()
plt.show()

print("All three measures capture uncertainty, but in slightly different ways")
print(f"Confidence: lower is more uncertain (min={confidences.min():.3f})")
print(f"Margin: lower is more uncertain (min={margins.min():.3f})")
print(f"Entropy: higher is more uncertain (max={entropies.max():.3f})")

## 4. Complete Active Learning Loop

Now let's build the full active learning loop:

1. Start with small labeled set
2. Train model
3. Select most uncertain samples from unlabeled pool
4. "Label" them (simulate oracle with true labels)
5. Add to labeled set
6. Repeat

We'll compare **active learning** vs **random sampling** to show the benefit.

### Active Learning Loop Implementation

In [ ]:
def active_learning_loop(
    query_strategy,
    initial_labeled,
    initial_unlabeled,
    n_iterations=20,
    n_query=50,
    epochs_per_iteration=10,
    strategy_name="Active"
):
    """Run active learning loop with given query strategy."""
    labeled_indices = initial_labeled.copy()
    unlabeled_indices = initial_unlabeled.copy()
    
    results = {
        'n_labeled': [],
        'accuracy': [],
        'train_loss': []
    }
    
    for iteration in range(n_iterations):
        # Train model on current labeled set
        model = SimpleCNN().to(device)
        loss = train_model(model, labeled_indices, epochs=epochs_per_iteration)
        
        # Evaluate
        accuracy = evaluate_model(model, test_loader)
        
        # Record results
        results['n_labeled'].append(len(labeled_indices))
        results['accuracy'].append(accuracy)
        results['train_loss'].append(loss)
        
        print(f"{strategy_name} - Iter {iteration+1}/{n_iterations}: "
              f"Labeled={len(labeled_indices)}, Acc={accuracy:.4f}")
        
        # Stop if no more unlabeled samples
        if len(unlabeled_indices) < n_query:
            break
        
        # Query strategy: select samples to label
        if query_strategy == 'random':
            # Random sampling baseline
            selected_indices = np.random.choice(unlabeled_indices, n_query, replace=False).tolist()
        else:
            # Use provided query strategy
            selected_indices, _ = query_strategy(model, unlabeled_indices, n_samples=n_query)
        
        # Simulate oracle: move selected samples from unlabeled to labeled
        labeled_indices.extend(selected_indices)
        unlabeled_indices = [idx for idx in unlabeled_indices if idx not in selected_indices]
    
    return results

print("Active learning loop ready")

### Run Active Learning with Entropy Sampling

Let's run the loop with entropy-based uncertainty sampling and compare with random sampling.

In [ ]:
# Reset random seed for reproducibility
set_seed(42)

# Run entropy-based active learning
print("Running entropy-based active learning...\n")
entropy_results = active_learning_loop(
    query_strategy=entropy_sampling,
    initial_labeled=labeled_indices.copy(),
    initial_unlabeled=unlabeled_indices.copy(),
    n_iterations=15,
    n_query=50,
    epochs_per_iteration=10,
    strategy_name="Entropy"
)

### Run Random Sampling Baseline

Random sampling is the baseline - it doesn't use any intelligence in selecting samples.

In [ ]:
# Reset random seed for fair comparison
set_seed(42)

# Run random sampling baseline
print("\nRunning random sampling baseline...\n")
random_results = active_learning_loop(
    query_strategy='random',
    initial_labeled=labeled_indices.copy(),
    initial_unlabeled=unlabeled_indices.copy(),
    n_iterations=15,
    n_query=50,
    epochs_per_iteration=10,
    strategy_name="Random"
)

### Visualize Learning Curves

The **learning curve** shows accuracy vs. number of labeled samples.

Active learning should reach higher accuracy with fewer labels.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(entropy_results['n_labeled'], entropy_results['accuracy'], 
         marker='o', label='Entropy Sampling (Active)', linewidth=2)
plt.plot(random_results['n_labeled'], random_results['accuracy'], 
         marker='s', label='Random Sampling (Baseline)', linewidth=2)

plt.xlabel('Number of Labeled Samples', fontsize=12)
plt.ylabel('Test Accuracy', fontsize=12)
plt.title('Active Learning: Entropy vs Random Sampling', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Compute label efficiency
target_acc = 0.90
entropy_n = next((n for n, acc in zip(entropy_results['n_labeled'], entropy_results['accuracy']) 
                  if acc >= target_acc), None)
random_n = next((n for n, acc in zip(random_results['n_labeled'], random_results['accuracy']) 
                 if acc >= target_acc), None)

if entropy_n and random_n:
    reduction = (random_n - entropy_n) / random_n * 100
    print(f"\nTo reach {target_acc:.1%} accuracy:")
    print(f"  Entropy sampling: {entropy_n} labels")
    print(f"  Random sampling: {random_n} labels")
    print(f"  Label reduction: {reduction:.1f}%")

### Key Insight

Active learning with entropy sampling **consistently outperforms** random sampling:

- Reaches target accuracy with **fewer labeled samples**
- More sample-efficient throughout training
- Especially valuable in early stages (when labels are most scarce)

This is the power of intelligent sample selection!

## 5. Query-by-Committee (QBC)

**Idea**: Train an **ensemble** of models and select samples where they **disagree** the most.

- More disagreement = more uncertainty
- Labeling resolves disagreement
- Ensemble provides better uncertainty estimates than single model

We measure disagreement using **vote entropy**.

### Implement Query-by-Committee

In [ ]:
def query_by_committee(models, unlabeled_indices, n_samples=10, batch_size=128):
    """Select samples where committee of models disagrees most."""
    # Set all models to eval mode
    for model in models:
        model.eval()
    
    unlabeled_subset = Subset(full_dataset, unlabeled_indices)
    unlabeled_loader = DataLoader(unlabeled_subset, batch_size=batch_size, shuffle=False)
    
    disagreements = []
    
    with torch.no_grad():
        for images, _ in unlabeled_loader:
            images = images.to(device)
            
            # Get predictions from all committee members
            predictions = []
            for model in models:
                outputs = model(images)
                probs = F.softmax(outputs, dim=1)
                predictions.append(probs)
            
            # Stack predictions: (n_models, batch_size, n_classes)
            predictions = torch.stack(predictions)
            
            # Average predictions across committee
            avg_probs = predictions.mean(dim=0)
            
            # Compute vote entropy
            log_probs = torch.log(avg_probs + 1e-10)
            batch_disagreement = -(avg_probs * log_probs).sum(dim=1)
            disagreements.extend(batch_disagreement.cpu().numpy())
    
    disagreements = np.array(disagreements)
    
    # Select samples with highest disagreement
    uncertain_indices = np.argsort(disagreements)[::-1][:n_samples]
    selected_indices = [unlabeled_indices[i] for i in uncertain_indices]
    
    return selected_indices, disagreements[uncertain_indices]

print("Query-by-committee ready")

### Train Committee of Models

We'll create diversity in the committee by:
1. Different random initializations
2. Different training subsets (bagging)
3. Different dropout rates

In [ ]:
def train_committee(labeled_indices, n_members=5, epochs=10):
    """Train committee of models with diversity."""
    committee = []
    
    for i in range(n_members):
        # Create model with different dropout
        dropout = 0.3 + i * 0.1  # 0.3, 0.4, 0.5, 0.6, 0.7
        model = SimpleCNN(dropout=dropout).to(device)
        
        # Bootstrap sample (sample with replacement)
        bootstrap_indices = np.random.choice(labeled_indices, 
                                            size=len(labeled_indices), 
                                            replace=True).tolist()
        
        # Train
        train_model(model, bootstrap_indices, epochs=epochs)
        committee.append(model)
        
        print(f"Trained committee member {i+1}/{n_members} (dropout={dropout:.1f})")
    
    return committee

# Train initial committee
committee = train_committee(labeled_indices, n_members=5, epochs=5)

### Visualize Committee Disagreement

Let's see where the committee disagrees on unlabeled samples.

In [ ]:
# Get predictions from committee on sample of unlabeled data
sample_indices = unlabeled_indices[:1000]
sample_subset = Subset(full_dataset, sample_indices)
sample_loader = DataLoader(sample_subset, batch_size=128, shuffle=False)

all_predictions = []
for model in committee:
    model.eval()
    predictions = []
    with torch.no_grad():
        for images, _ in sample_loader:
            images = images.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            predictions.extend(preds.cpu().numpy())
    all_predictions.append(predictions)

all_predictions = np.array(all_predictions)  # (n_models, n_samples)

# Compute disagreement rate
disagreement_rates = []
for i in range(all_predictions.shape[1]):
    # How many unique predictions for this sample?
    unique_preds = len(np.unique(all_predictions[:, i]))
    # Disagreement = (unique - 1) / (n_models - 1)
    disagreement_rates.append((unique_preds - 1) / (len(committee) - 1))

disagreement_rates = np.array(disagreement_rates)

plt.figure(figsize=(10, 5))
plt.hist(disagreement_rates, bins=20, alpha=0.7, edgecolor='black')
plt.xlabel('Disagreement Rate', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.title('Committee Disagreement Distribution', fontsize=14, fontweight='bold')
plt.axvline(disagreement_rates.mean(), color='red', linestyle='--', 
           label=f'Mean: {disagreement_rates.mean():.3f}')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Samples with full agreement: {(disagreement_rates == 0).sum()}")
print(f"Samples with full disagreement: {(disagreement_rates == 1).sum()}")
print(f"Samples with partial disagreement: {((disagreement_rates > 0) & (disagreement_rates < 1)).sum()}")

### Run QBC Active Learning

Now let's run the full active learning loop with query-by-committee.

In [ ]:
def qbc_active_learning_loop(
    initial_labeled,
    initial_unlabeled,
    n_iterations=15,
    n_query=50,
    epochs_per_iteration=10,
    n_committee=5
):
    """Active learning with query-by-committee."""
    labeled_indices = initial_labeled.copy()
    unlabeled_indices = initial_unlabeled.copy()
    
    results = {
        'n_labeled': [],
        'accuracy': [],
    }
    
    for iteration in range(n_iterations):
        # Train committee
        committee = train_committee(labeled_indices, n_members=n_committee, 
                                   epochs=epochs_per_iteration)
        
        # Evaluate first committee member (as representative)
        accuracy = evaluate_model(committee[0], test_loader)
        
        results['n_labeled'].append(len(labeled_indices))
        results['accuracy'].append(accuracy)
        
        print(f"QBC - Iter {iteration+1}/{n_iterations}: "
              f"Labeled={len(labeled_indices)}, Acc={accuracy:.4f}")
        
        if len(unlabeled_indices) < n_query:
            break
        
        # Query-by-committee
        selected_indices, _ = query_by_committee(committee, unlabeled_indices, 
                                                n_samples=n_query)
        
        # Update sets
        labeled_indices.extend(selected_indices)
        unlabeled_indices = [idx for idx in unlabeled_indices if idx not in selected_indices]
    
    return results

# Run QBC (this will take longer due to ensemble training)
set_seed(42)
print("Running query-by-committee active learning...\n")
qbc_results = qbc_active_learning_loop(
    initial_labeled=labeled_indices.copy(),
    initial_unlabeled=unlabeled_indices.copy(),
    n_iterations=10,  # Fewer iterations due to computational cost
    n_query=50,
    epochs_per_iteration=5,
    n_committee=3  # Smaller committee for speed
)

### Compare All Strategies

Let's plot all three strategies together.

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(entropy_results['n_labeled'], entropy_results['accuracy'], 
         marker='o', label='Entropy Sampling', linewidth=2)
plt.plot(qbc_results['n_labeled'], qbc_results['accuracy'], 
         marker='^', label='Query-by-Committee', linewidth=2)
plt.plot(random_results['n_labeled'], random_results['accuracy'], 
         marker='s', label='Random Sampling', linewidth=2, linestyle='--')

plt.xlabel('Number of Labeled Samples', fontsize=12)
plt.ylabel('Test Accuracy', fontsize=12)
plt.title('Active Learning: Strategy Comparison', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nAll active learning strategies outperform random sampling!")
print("QBC can provide additional benefits with proper ensemble diversity.")

## 6. Batch Mode Active Learning

So far we've selected samples one at a time (or in small batches).

**Problem**: In practice, we want to label **batches** of samples (e.g., 100 at once).

**Challenge**: Selecting top-100 most uncertain samples can be **redundant** - they might all be similar!

**Solution**: Balance uncertainty with **diversity** to cover more of the data space.

### K-Means Diversity Sampling

**Idea**: 
1. Compute uncertainty scores
2. Filter to top-K most uncertain samples
3. Cluster in feature space using K-means
4. Select most uncertain sample from each cluster

This ensures both **high uncertainty** and **high diversity**.

In [ ]:
def diverse_uncertainty_sampling(model, unlabeled_indices, n_samples=10, 
                                oversample_factor=10, batch_size=128):
    """Select diverse uncertain samples using K-means clustering."""
    model.eval()
    
    unlabeled_subset = Subset(full_dataset, unlabeled_indices)
    unlabeled_loader = DataLoader(unlabeled_subset, batch_size=batch_size, shuffle=False)
    
    # Get uncertainties and features
    entropies = []
    features = []
    
    with torch.no_grad():
        for images, _ in unlabeled_loader:
            images = images.to(device)
            outputs, feats = model(images, return_features=True)
            probs = F.softmax(outputs, dim=1)
            
            # Entropy
            log_probs = torch.log(probs + 1e-10)
            batch_entropies = -(probs * log_probs).sum(dim=1)
            entropies.extend(batch_entropies.cpu().numpy())
            
            # Features for diversity
            features.append(feats.cpu().numpy())
    
    entropies = np.array(entropies)
    features = np.concatenate(features, axis=0)
    
    # Step 1: Filter to most uncertain samples
    n_oversample = min(n_samples * oversample_factor, len(unlabeled_indices))
    uncertain_mask = np.argsort(entropies)[::-1][:n_oversample]
    
    # Step 2: Cluster uncertain samples
    uncertain_features = features[uncertain_mask]
    kmeans = KMeans(n_clusters=n_samples, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(uncertain_features)
    
    # Step 3: Select most uncertain sample from each cluster
    selected_local_indices = []
    for cluster_id in range(n_samples):
        cluster_mask = cluster_labels == cluster_id
        cluster_indices = np.where(cluster_mask)[0]
        
        if len(cluster_indices) > 0:
            # Get entropies for this cluster
            cluster_entropies = entropies[uncertain_mask[cluster_indices]]
            # Select highest entropy in cluster
            best_in_cluster = cluster_indices[np.argmax(cluster_entropies)]
            selected_local_indices.append(uncertain_mask[best_in_cluster])
    
    # Map back to global indices
    selected_indices = [unlabeled_indices[i] for i in selected_local_indices]
    
    return selected_indices, entropies[selected_local_indices]

print("Diverse uncertainty sampling ready")

### Visualize Diversity

Let's compare samples selected by pure uncertainty vs. diverse uncertainty.

In [ ]:
# Train a model
model = SimpleCNN().to(device)
train_model(model, labeled_indices, epochs=5)

# Select samples with both strategies
pure_uncertain, _ = entropy_sampling(model, unlabeled_indices, n_samples=20)
diverse_uncertain, _ = diverse_uncertainty_sampling(model, unlabeled_indices, 
                                                   n_samples=20, oversample_factor=10)

# Get images
def get_images(indices):
    images = []
    for idx in indices:
        img, _ = full_dataset[idx]
        images.append(img)
    return torch.stack(images)

pure_images = get_images(pure_uncertain)
diverse_images = get_images(diverse_uncertain)

# Plot
fig, axes = plt.subplots(2, 10, figsize=(15, 3))

for i in range(10):
    axes[0, i].imshow(pure_images[i].squeeze(), cmap='gray')
    axes[0, i].axis('off')
    axes[1, i].imshow(diverse_images[i].squeeze(), cmap='gray')
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Pure\nUncertainty', fontsize=10, rotation=0, ha='right')
axes[1, 0].set_ylabel('Diverse\nUncertainty', fontsize=10, rotation=0, ha='right')

plt.suptitle('Sample Selection: Pure vs Diverse Uncertainty', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("Notice: Diverse sampling tends to cover more digit classes!")

## 7. Deep Active Learning with MC Dropout

**Monte Carlo (MC) Dropout** is a technique to estimate **uncertainty in neural networks**.

**Idea**:
1. Keep dropout enabled during inference
2. Run multiple forward passes (stochastic)
3. Get different predictions each time
4. Variance across predictions = uncertainty

This is a form of **Bayesian approximation** - treating dropout as Bayesian inference.

### Implement MC Dropout Uncertainty

In [ ]:
def mc_dropout_uncertainty(model, unlabeled_indices, n_samples=10, 
                          n_forward_passes=10, batch_size=128):
    """Estimate uncertainty using Monte Carlo dropout."""
    # Enable dropout during inference
    model.train()  # This keeps dropout active!
    
    unlabeled_subset = Subset(full_dataset, unlabeled_indices)
    unlabeled_loader = DataLoader(unlabeled_subset, batch_size=batch_size, shuffle=False)
    
    all_predictions = []
    
    # Run multiple forward passes with dropout
    with torch.no_grad():
        for _ in range(n_forward_passes):
            predictions = []
            for images, _ in unlabeled_loader:
                images = images.to(device)
                outputs = model(images)
                probs = F.softmax(outputs, dim=1)
                predictions.append(probs.cpu())
            predictions = torch.cat(predictions, dim=0)
            all_predictions.append(predictions)
    
    # Stack: (n_forward_passes, n_samples, n_classes)
    all_predictions = torch.stack(all_predictions)
    
    # Compute predictive entropy (BALD)
    # Average predictions
    mean_probs = all_predictions.mean(dim=0)
    
    # Entropy of mean
    log_mean_probs = torch.log(mean_probs + 1e-10)
    entropy_of_mean = -(mean_probs * log_mean_probs).sum(dim=1)
    
    uncertainties = entropy_of_mean.numpy()
    
    # Select most uncertain
    uncertain_indices = np.argsort(uncertainties)[::-1][:n_samples]
    selected_indices = [unlabeled_indices[i] for i in uncertain_indices]
    
    return selected_indices, uncertainties[uncertain_indices]

print("MC Dropout uncertainty ready")

### Compare MC Dropout with Standard Uncertainty

Let's see how MC Dropout uncertainty differs from standard entropy.

In [ ]:
# Get uncertainties with both methods
model = SimpleCNN(dropout=0.5).to(device)
train_model(model, labeled_indices, epochs=5)

# Standard entropy
_, standard_unc = entropy_sampling(model, unlabeled_indices[:1000], n_samples=1000)

# MC Dropout
_, mc_unc = mc_dropout_uncertainty(model, unlabeled_indices[:1000], 
                                  n_samples=1000, n_forward_passes=10)

# Plot comparison
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(standard_unc, bins=50, alpha=0.7, edgecolor='black')
plt.xlabel('Uncertainty', fontsize=11)
plt.ylabel('Count', fontsize=11)
plt.title('Standard Entropy', fontsize=12, fontweight='bold')
plt.axvline(standard_unc.mean(), color='red', linestyle='--', 
           label=f'Mean: {standard_unc.mean():.3f}')
plt.legend()

plt.subplot(1, 2, 2)
plt.hist(mc_unc, bins=50, alpha=0.7, edgecolor='black')
plt.xlabel('Uncertainty', fontsize=11)
plt.ylabel('Count', fontsize=11)
plt.title('MC Dropout (10 passes)', fontsize=12, fontweight='bold')
plt.axvline(mc_unc.mean(), color='red', linestyle='--', 
           label=f'Mean: {mc_unc.mean():.3f}')
plt.legend()

plt.tight_layout()
plt.show()

print("MC Dropout provides a different uncertainty estimate by:")
print("  - Averaging over multiple stochastic forward passes")
print("  - Capturing model uncertainty (not just data uncertainty)")
print("  - Better calibrated for out-of-distribution samples")

## 8. Evaluation Metrics

How do we measure the effectiveness of active learning?

Key metrics:

1. **Learning Curve**: Accuracy vs. number of labeled samples
2. **Area Under Learning Curve (AULC)**: Total area under the curve
3. **Label Efficiency**: How many labels to reach target accuracy
4. **Reduction Rate**: (Random labels - Active labels) / Random labels

### Compute Area Under Learning Curve

In [ ]:
def compute_aulc(n_labeled, accuracies):
    """Compute area under learning curve using trapezoidal rule."""
    # Normalize by x-axis range
    aulc = np.trapz(accuracies, n_labeled)
    normalized_aulc = aulc / (n_labeled[-1] - n_labeled[0])
    return aulc, normalized_aulc

# Compute AULC for each strategy
entropy_aulc, entropy_norm_aulc = compute_aulc(
    entropy_results['n_labeled'], 
    entropy_results['accuracy']
)
random_aulc, random_norm_aulc = compute_aulc(
    random_results['n_labeled'], 
    random_results['accuracy']
)

print("Area Under Learning Curve (AULC):\n")
print(f"Entropy sampling: {entropy_norm_aulc:.4f}")
print(f"Random sampling:  {random_norm_aulc:.4f}")
print(f"\nImprovement: {(entropy_norm_aulc - random_norm_aulc) / random_norm_aulc * 100:.2f}%")
print("\nHigher AULC = better overall performance across all label budgets")

### Label Efficiency Analysis

At different accuracy targets, how many labels does each strategy need?

In [ ]:
def find_labels_for_accuracy(n_labeled, accuracies, target_acc):
    """Find number of labels needed to reach target accuracy."""
    for n, acc in zip(n_labeled, accuracies):
        if acc >= target_acc:
            return n
    return None

# Test multiple accuracy targets
targets = [0.85, 0.87, 0.89, 0.91]

print("Label Efficiency Comparison:\n")
print(f"{'Target Acc':<12} {'Entropy':<12} {'Random':<12} {'Reduction':<12}")
print("-" * 50)

for target in targets:
    entropy_n = find_labels_for_accuracy(
        entropy_results['n_labeled'], 
        entropy_results['accuracy'], 
        target
    )
    random_n = find_labels_for_accuracy(
        random_results['n_labeled'], 
        random_results['accuracy'], 
        target
    )
    
    if entropy_n and random_n:
        reduction = (random_n - entropy_n) / random_n * 100
        print(f"{target:.2f}         {entropy_n:<12} {random_n:<12} {reduction:.1f}%")
    else:
        print(f"{target:.2f}         {'N/A':<12} {'N/A':<12} {'N/A':<12}")

print("\nActive learning consistently needs fewer labels to reach any target accuracy!")

## 9. Practical Considerations

Active learning in the real world has several challenges:

1. **Cold Start Problem**: How to initialize with very few labels?
2. **Stopping Criteria**: When to stop querying?
3. **Computational Cost**: Query strategies can be expensive
4. **Oracle Quality**: Human annotators make mistakes
5. **Class Imbalance**: May oversample minority classes

Let's explore some of these.

### Cold Start: Initial Labeled Set

The quality of the initial labeled set matters!

Strategies:
- **Random**: Simple but may miss rare classes
- **Stratified**: Ensure all classes represented
- **Diverse**: Use K-means on features (unsupervised)

Let's test the impact of different initialization strategies.

In [ ]:
def stratified_init(dataset, n_samples_per_class=10):
    """Create stratified initial labeled set with equal samples per class."""
    indices_by_class = defaultdict(list)
    
    # Group indices by class
    for idx in range(len(dataset)):
        _, label = dataset[idx]
        indices_by_class[label].append(idx)
    
    # Sample equally from each class
    stratified_indices = []
    for label in sorted(indices_by_class.keys()):
        class_indices = indices_by_class[label]
        sampled = np.random.choice(class_indices, 
                                  size=min(n_samples_per_class, len(class_indices)), 
                                  replace=False)
        stratified_indices.extend(sampled.tolist())
    
    return stratified_indices

# Create stratified initial set
stratified_labeled = stratified_init(full_dataset, n_samples_per_class=10)

# Check class distribution
random_labels = [full_dataset[idx][1] for idx in labeled_indices]
stratified_labels = [full_dataset[idx][1] for idx in stratified_labeled]

random_counts = np.bincount(random_labels, minlength=10)
stratified_counts = np.bincount(stratified_labels, minlength=10)

# Plot
x = np.arange(10)
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width/2, random_counts, width, label='Random Init', alpha=0.8)
ax.bar(x + width/2, stratified_counts, width, label='Stratified Init', alpha=0.8)

ax.set_xlabel('Class', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title('Initial Labeled Set: Random vs Stratified', fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("Stratified initialization ensures all classes are represented equally.")
print("This prevents the model from being initially biased toward majority classes.")

### Stopping Criteria

When should we stop active learning?

Options:
1. **Fixed budget**: Stop after N labels
2. **Performance threshold**: Stop when accuracy > target
3. **Plateau detection**: Stop when accuracy stops improving
4. **Uncertainty collapse**: Stop when all samples are confident

Let's implement plateau detection.

In [ ]:
def detect_plateau(accuracies, window=3, threshold=0.001):
    """
    Detect if accuracy has plateaued.
    
    Returns True if average improvement over last 'window' iterations
    is below threshold.
    """
    if len(accuracies) < window + 1:
        return False
    
    recent_accs = accuracies[-window-1:]
    improvements = [recent_accs[i+1] - recent_accs[i] for i in range(len(recent_accs)-1)]
    avg_improvement = np.mean(improvements)
    
    return avg_improvement < threshold

# Test on our entropy results
for i in range(len(entropy_results['accuracy'])):
    accs = entropy_results['accuracy'][:i+1]
    is_plateau = detect_plateau(accs, window=3, threshold=0.005)
    if is_plateau:
        print(f"Plateau detected at iteration {i+1}")
        print(f"  Accuracy: {accs[-1]:.4f}")
        print(f"  Labels used: {entropy_results['n_labeled'][i]}")
        break

print("\nPlateau detection can save annotation budget by stopping early.")

## 10. Applications and Use Cases

Active learning shines in domains with **expensive annotations**:

### Medical Imaging
- Expert radiologists cost $100-200/hour
- Each scan may take 10-30 minutes to annotate
- Active learning can reduce labeling costs by 50-90%

### Autonomous Driving
- Need to handle rare edge cases
- Active learning finds unusual scenarios
- Improves safety-critical performance

### Domain-Specific NLP
- Legal, medical, scientific text
- Requires domain experts
- Active learning maximizes expert time

### Anomaly Detection
- Anomalies are rare by definition
- Active learning finds boundary cases
- Critical for security applications

### Simulate Class Imbalance Scenario

Active learning can help with imbalanced datasets by focusing on minority classes.

In [ ]:
# Create imbalanced dataset: keep more samples from classes 0-4, fewer from 5-9
imbalanced_indices = []
for idx in range(len(full_dataset)):
    _, label = full_dataset[idx]
    # Keep 100% of classes 0-4, only 10% of classes 5-9
    if label < 5 or np.random.rand() < 0.1:
        imbalanced_indices.append(idx)

print(f"Imbalanced dataset size: {len(imbalanced_indices)} (from {len(full_dataset)})")

# Check class distribution
imbalanced_labels = [full_dataset[idx][1] for idx in imbalanced_indices]
imbalanced_counts = np.bincount(imbalanced_labels, minlength=10)

plt.figure(figsize=(10, 5))
plt.bar(range(10), imbalanced_counts, alpha=0.7, edgecolor='black')
plt.xlabel('Class', fontsize=11)
plt.ylabel('Count', fontsize=11)
plt.title('Imbalanced Dataset Distribution', fontsize=12, fontweight='bold')
plt.axhline(imbalanced_counts.mean(), color='red', linestyle='--', 
           label=f'Mean: {imbalanced_counts.mean():.0f}')
plt.legend()
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("\nActive learning on imbalanced data:")
print("  - Uncertainty sampling naturally focuses on minority classes")
print("  - Model is uncertain about underrepresented classes")
print("  - This helps balance the dataset over time")

## Key Takeaways

### Core Concepts

1. **Active learning intelligently selects which data to label** - dramatically reducing annotation costs

2. **Query strategies** determine sample selection:
   - **Uncertainty sampling**: least confidence, margin, entropy
   - **Query-by-committee**: ensemble disagreement
   - **Diversity sampling**: cover data space

3. **The active learning loop**:
   - Train on labeled set
   - Query most informative samples
   - Oracle labels them
   - Repeat

### Practical Insights

4. **Label efficiency**: Active learning can reduce labeling needs by **10-100x**

5. **Batch mode**: Balance uncertainty with diversity to avoid redundant samples

6. **Deep active learning**: Use MC Dropout or ensemble methods for better uncertainty estimates

7. **Real-world challenges**:
   - Cold start with stratified sampling
   - Stop early with plateau detection
   - Handle class imbalance naturally

### When to Use Active Learning

8. **Perfect for**:
   - Expensive annotations (medical, legal, domain expertise)
   - Large unlabeled datasets
   - Limited annotation budget
   - Rare events / edge cases

9. **Not worth it for**:
   - Cheap annotations
   - Very small datasets
   - Already labeled data available

## Summary

Active learning is a **paradigm shift** in machine learning:

**Traditional ML**: "Label all the data, then train"

**Active Learning**: "Train, query, label intelligently, repeat"

By selecting the **most informative samples**, active learning achieves comparable performance with a **fraction of the labels**.

This makes machine learning feasible in domains where annotation is the bottleneck - from medical imaging to autonomous systems to domain-specific NLP.

**The key insight**: Not all data points are equal. Smart selection beats random sampling every time.